# Notebook 02f — Alignment Diagnostic (02c ↔ 02e)

**Purpose.** Verify that the LLM classification setup used in Notebook 02c (original scoring of 4,500 comments) is byte-identical to the setup used in Notebook 02e (test-retest re-scoring of 100 comments). If any of the three components — `SYSTEM_PROMPT`, the `classify_one` function source, or the `MODEL` constant — differ between the two notebooks, the test-retest kappa numbers reported in 02e would be measuring a mix of stochastic variation *and* setup drift, which is not what we want.

**What this notebook checks.**

1. `SYSTEM_PROMPT` — the system-prompt string used in the classifier.
2. `classify_one` — the function that sends one comment to the API and parses the response.
3. `MODEL` — the Anthropic model name used in the classification loop.

**Expected outcome.** All three components should report `IDENTICAL`. If any report `DIFFERS`, the offending diff is printed for inspection so the source of drift can be identified.

**Runtime.** ~1 second. No API calls, no file writes.

## 0. Setup — load both notebooks

In [ ]:
import json
import re
import difflib
from pathlib import Path

# Both notebooks are expected in the working directory.
PATH_02C = Path('02c_llm_classification.ipynb')
PATH_02E = Path('02e_test_retest.ipynb')

for p in [PATH_02C, PATH_02E]:
    assert p.exists(), (
        f'Cannot find {p}. Place this notebook in the same directory '
        'as 02c and 02e, or edit the paths above.'
    )

nb_02c = json.loads(PATH_02C.read_text())
nb_02e = json.loads(PATH_02E.read_text())

print(f'Loaded {PATH_02C.name}: {len(nb_02c["cells"])} cells')
print(f'Loaded {PATH_02E.name}: {len(nb_02e["cells"])} cells')

## 1. Extract the three components from each notebook

The helpers below scan each notebook's code cells for the target definitions and return their source, normalised so leading/trailing whitespace and quote-style variations do not produce false differences.

In [ ]:
def all_code_source(nb):
    """Concatenate the source of every code cell in a notebook."""
    chunks = []
    for c in nb['cells']:
        if c['cell_type'] != 'code':
            continue
        src = ''.join(c['source']) if isinstance(c['source'], list) \
              else c['source']
        chunks.append(src)
    return '\n\n# --- cell boundary ---\n\n'.join(chunks)


def extract_system_prompt(source):
    """Extract the SYSTEM_PROMPT triple-quoted string content."""
    m = re.search(
        r"SYSTEM_PROMPT\s*=\s*(?:'''|\"\"\")(.*?)(?:'''|\"\"\")",
        source, flags=re.DOTALL,
    )
    if not m:
        return None
    return m.group(1).strip()


def extract_function(source, func_name):
    """Extract a function definition body from source.

    We take the `def <name>(...):` line and consume subsequent lines
    until we hit a line that starts at column 0 with something other
    than whitespace or `#` (a common heuristic for end-of-function).
    """
    lines = source.splitlines(keepends=True)
    n = len(lines)
    for i, line in enumerate(lines):
        if re.match(rf'^def\s+{func_name}\b', line):
            body = [line]
            j = i + 1
            while j < n:
                ln = lines[j]
                # blank line — keep going
                if ln.strip() == '':
                    body.append(ln)
                    j += 1
                    continue
                # continued function body (indented)
                if ln.startswith((' ', '\t')):
                    body.append(ln)
                    j += 1
                    continue
                # top-level line — function has ended
                break
            return ''.join(body).rstrip() + '\n'
    return None


def extract_model_constant(source):
    """Extract the MODEL = '...' constant used in the classification loop.

    Prefer a top-level `MODEL = '...'` line; fall back to the default
    argument of classify_one if MODEL is not defined at module level.
    """
    m = re.search(
        r"^\s*MODEL\s*=\s*[\"']([^\"']+)[\"']",
        source, flags=re.MULTILINE,
    )
    if m:
        return m.group(1)
    m = re.search(
        r"def\s+classify_one\([^)]*?model\s*:\s*str\s*=\s*[\"']([^\"']+)[\"']",
        source, flags=re.DOTALL,
    )
    if m:
        return m.group(1)
    return None


src_02c = all_code_source(nb_02c)
src_02e = all_code_source(nb_02e)
print(f'02c total code size: {len(src_02c):,} chars')
print(f'02e total code size: {len(src_02e):,} chars')

## 2. Compare the three components byte-by-byte

For each of `SYSTEM_PROMPT`, `classify_one`, and `MODEL`, print either `IDENTICAL` or `DIFFERS` with a unified diff showing the offending lines.

In [ ]:
import ast

def ast_equivalent(src_a, src_b):
    """Return True if two Python source strings parse to the same AST.

    This ignores cosmetic differences that Python treats as equivalent:
    single vs double quotes, whitespace around colons in dict literals,
    docstring additions/removals, and trailing whitespace.
    """
    try:
        tree_a = ast.parse(src_a)
        tree_b = ast.parse(src_b)
        # Strip docstrings from both trees so a documentation-only
        # difference doesn't fail the check.
        for tree in (tree_a, tree_b):
            for node in ast.walk(tree):
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef,
                                       ast.ClassDef, ast.Module)):
                    if (node.body and isinstance(node.body[0], ast.Expr)
                            and isinstance(node.body[0].value, ast.Constant)
                            and isinstance(node.body[0].value.value, str)):
                        node.body = node.body[1:]
        return ast.dump(tree_a) == ast.dump(tree_b)
    except SyntaxError:
        return None  # can't parse — fall back to string compare


def compare(label, val_02c, val_02e, use_ast=False,
             show_full_on_mismatch=True):
    print('=' * 78)
    print(f'  {label}')
    print('=' * 78)
    if val_02c is None or val_02e is None:
        print(f'  [ERROR] could not extract from one of the notebooks:')
        print(f'    02c: {"OK" if val_02c is not None else "NOT FOUND"}')
        print(f'    02e: {"OK" if val_02e is not None else "NOT FOUND"}')
        return False

    byte_identical = (val_02c == val_02e)
    ast_identical = None
    if use_ast and not byte_identical:
        ast_identical = ast_equivalent(val_02c, val_02e)

    if byte_identical:
        print(f'  [IDENTICAL] length = {len(val_02c):,} chars')
        return True
    if ast_identical is True:
        print(f'  [FUNCTIONALLY IDENTICAL] '
              f'02c = {len(val_02c):,} chars, '
              f'02e = {len(val_02e):,} chars')
        print('  Byte-level differences are cosmetic only '
              '(quote-style, whitespace, docstring). Python AST '
              'confirms the two are semantically identical — '
              'they behave exactly the same way at runtime.')
        return True

    print(f'  [DIFFERS] 02c = {len(val_02c):,} chars, '
          f'02e = {len(val_02e):,} chars')
    if ast_identical is False:
        print('  AST comparison also shows a semantic difference — '
              'this is a real, functional divergence, not just cosmetic.')
    if show_full_on_mismatch:
        diff = difflib.unified_diff(
            val_02c.splitlines(keepends=True),
            val_02e.splitlines(keepends=True),
            fromfile='02c', tofile='02e', n=2,
        )
        print('\n  Unified diff:')
        for line in diff:
            print('   ' + line.rstrip('\n'))
    return False


sp_02c = extract_system_prompt(src_02c)
sp_02e = extract_system_prompt(src_02e)
fn_02c = extract_function(src_02c, 'classify_one')
fn_02e = extract_function(src_02e, 'classify_one')
mdl_02c = extract_model_constant(src_02c)
mdl_02e = extract_model_constant(src_02e)

results = []
results.append(compare('SYSTEM_PROMPT (verbatim string)',
                       sp_02c, sp_02e, use_ast=False))
results.append(compare('classify_one (function source)',
                       fn_02c, fn_02e, use_ast=True))
results.append(compare('MODEL (classification model constant)',
                       mdl_02c, mdl_02e, use_ast=False,
                       show_full_on_mismatch=False))

print('=' * 78)
print('  OVERALL')
print('=' * 78)
if all(results):
    print('  All three components match between 02c and 02e '
          '(byte-identical or semantically identical).')
    print('  The test-retest kappa numbers reported in 02e reflect')
    print('  genuine LLM-vs-LLM stochasticity, not setup drift.')
else:
    print('  One or more components DIFFER between the notebooks '
          'in a way that changes behaviour.')
    print('  See the diff(s) above and reconcile before trusting the')
    print('  test-retest kappa numbers as a pure stability measure.')

## 3. Model constant — side-by-side confirmation

For an at-a-glance sanity check, print the extracted model constants and the API-parameter defaults from both notebooks.

In [ ]:
print(f'02c MODEL: {mdl_02c!r}')
print(f'02e MODEL: {mdl_02e!r}')
print()

# Also confirm the max_tokens used in the API call is the same
def extract_max_tokens(source):
    m = re.search(r'max_tokens\s*=\s*(\d+)', source)
    return int(m.group(1)) if m else None

# We look inside the classify_one function source specifically, so we
# don't pick up the max_tokens=20 in the API smoke test.
mt_02c = extract_max_tokens(fn_02c) if fn_02c else None
mt_02e = extract_max_tokens(fn_02e) if fn_02e else None
print(f'02c classify_one max_tokens: {mt_02c}')
print(f'02e classify_one max_tokens: {mt_02e}')
print()

if mdl_02c == mdl_02e and mt_02c == mt_02e:
    print('Both notebooks use the same model and max_tokens for classification.')
else:
    print('WARNING: model or max_tokens differ. Investigate.')

## 4. Interpreting the diagnostic result

If the overall verdict in §2 reports **all identical**, this notebook serves as programmatic evidence that the test-retest was conducted under a fixed setup. A short footnote summarising this alignment can be included in the Methods chapter:

> *"Alignment between the original classification (Notebook 02c) and the test-retest re-classification (Notebook 02e) was verified programmatically (Notebook 02f). The system prompt, the API-call function, and the model constant are byte-identical between the two notebooks; the reported kappa therefore reflects the classifier's self-consistency under a fixed setup rather than any incidental drift in configuration."*

If any component reports **DIFFERS**, the unified diff printed in §2 identifies where the two notebooks disagree. Common causes:

- A prompt edit made in one notebook but not synced to the other (most common for `SYSTEM_PROMPT`).
- A refactor of `classify_one` (e.g., added retry logic in one, not the other).
- A model-constant change made when experimenting with a different model.

Once identified, the two notebooks should be re-synchronised and Notebook 02e re-run. The resulting test-retest kappa is then a clean measurement of the classifier's self-consistency.
